# ✈️ Обнаружение самолётов — обучение на Colab

**Порядок запуска:** выполняй ячейки сверху вниз по одной.

Перед запуском убедись что выбран GPU:  
`Среда выполнения → Сменить среду выполнения → T4 GPU`

## 1. Проверка GPU

In [ ]:
import torch

print('CUDA доступна:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print(f'Память GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('GPU не найден! Зайди в Среда выполнения -> Сменить среду -> T4 GPU')

## 2. Монтирование Google Drive

Чекпоинты будут сохраняться на Drive — не потеряются при срыве сессии.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/airplane-detection'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Папка на Drive: {DRIVE_DIR}')

## 3. Клонирование репозитория

Замени `REPO_URL` на ссылку своего репозитория.

In [ ]:
REPO_URL = 'https://github.com/ВАШ_ЮЗЕРНЕЙМ/airplane-detection.git'  # <-- замени!

import subprocess, os
from pathlib import Path

%cd /content

# Клонируем в фиксированную папку 'project' чтобы имя не зависело от GitHub
if Path('/content/project').exists():
    print('Папка уже существует, пропускаем clone')
else:
    !git clone {REPO_URL} project

%cd /content/project
!ls

## 4. Установка зависимостей

In [ ]:
%cd /content/project

!pip install -q -r requirements.txt
!apt-get install -y -q p7zip-full
print('Зависимости установлены')

## 5. Скачивание и распаковка датасета

HRPlanes с Zenodo (~10 GB). Займёт около 5–10 минут.

In [ ]:
%cd /content/project

!python scripts/download.py

In [ ]:
# Проверяем что датасет лежит правильно
from pathlib import Path

ok = True
for split in ('train', 'val', 'test'):
    imgs = list(Path(f'data/hrplanes/{split}/images').glob('*.jpg'))
    lbls = list(Path(f'data/hrplanes/{split}/labels').glob('*.txt'))
    status = 'OK' if imgs and lbls else 'ОШИБКА'
    print(f'[{status}] {split:5}: {len(imgs):5} изображений, {len(lbls):5} меток')
    if not imgs:
        ok = False

if ok:
    print('Датасет готов к обучению')
else:
    print('Что-то пошло не так — проверь вывод download.py выше')

## 6. Настройка сохранения чекпоинтов на Google Drive

In [ ]:
import os, shutil
from pathlib import Path

%cd /content/project

# Создаём папки на Drive
models_dir = f'{DRIVE_DIR}/models'
os.makedirs(f'{models_dir}/epochs', exist_ok=True)

# Символическая ссылка: assets/models -> Drive
local_models = Path('assets/models')
local_models.parent.mkdir(parents=True, exist_ok=True)

if local_models.exists() and not local_models.is_symlink():
    shutil.rmtree(local_models)

if not local_models.exists():
    os.symlink(models_dir, str(local_models))

print(f'Чекпоинты -> {models_dir}')
print('  best.pt             — лучший по mAP50')
print('  last.pt             — последняя эпоха')
print('  epochs/epoch_NNN.pt — каждая эпоха отдельно')

## 7. Smoke-test — 3 эпохи

Убеждаемся что всё работает (~5 мин) перед полным обучением.

In [ ]:
import yaml

%cd /content/project

with open('configs/config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

import copy
cfg_smoke = copy.deepcopy(cfg)
cfg_smoke['training']['epochs'] = 3

with open('configs/config_smoke.yaml', 'w') as f:
    yaml.dump(cfg_smoke, f, allow_unicode=True)

print('Smoke-test: 3 эпохи')

In [ ]:
%cd /content/project
!python scripts/train.py --config configs/config_smoke.yaml

## 8. Полное обучение

Если smoke-test прошёл без ошибок — запускаем полное обучение.

**60 эпох** гарантированно влезают в 2 часа на T4.

In [ ]:
import yaml, copy

%cd /content/project

EPOCHS = 60  # измени на 100 если времени больше

with open('configs/config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

cfg_full = copy.deepcopy(cfg)
cfg_full['training']['epochs'] = EPOCHS

with open('configs/config_full.yaml', 'w') as f:
    yaml.dump(cfg_full, f, allow_unicode=True)

print(f'Запускаем обучение на {EPOCHS} эпох...')

In [ ]:
%cd /content/project
!python scripts/train.py --config configs/config_full.yaml

## 9. Оценка на val и test

In [ ]:
%cd /content/project
!python scripts/val.py  --config configs/config_full.yaml --ckpt assets/models/best.pt

In [ ]:
%cd /content/project
!python scripts/test.py --config configs/config_full.yaml --ckpt assets/models/best.pt

## 10. Список сохранённых чекпоинтов

In [ ]:
from pathlib import Path

print('Чекпоинты на Google Drive:')
for f in sorted(Path('assets/models').rglob('*.pt')):
    size_mb = f.stat().st_size / 1e6
    print(f'  {str(f):<50} {size_mb:.1f} MB')